# Amazon Categorizer Playground

Demonstrates the categorizer API against synthetic data.  No real Amazon CSV needed.
All outputs go to a temporary directory so nothing lands in `data/`, `out/`, or `state/`.

For real use replace `workspace` with the actual paths:
```python
state_dir = Path('state')
output_dir = Path('out')
input_csv  = Path('data/amazon_2026_01.csv')
```

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

from analysis.amazon.notebook import (
    load_categorization_state,
    load_events_dataframe,
    run_amazon_categorization,
)
from analysis.log import configure_logging
from analysis.storage import JsonStateStore

# Isolated workspace for this notebook run
workspace = Path(tempfile.mkdtemp(prefix='amazon_playground_'))
state_dir  = workspace / 'state'
output_dir = workspace / 'out'
input_csv  = workspace / 'amazon_jan.csv'

# Stream logging only (no file handler needed for interactive exploration)
configure_logging()

print('Workspace:', workspace)

## Synthetic input CSV

Mix of items that will be pre-seeded as known and items that will be left uncategorized.

In [ ]:
raw = pd.DataFrame({
    'Description': [
        'Echo Dot (3rd Gen)',
        'Paper Towels 12-Pack',
        'USB-C Cable 6ft',
        'Paper Towels 12-Pack',   # duplicate — should reuse mapping
        'Coffee Pods 36ct',
        'Unknown Gadget XYZ',
    ],
    'Amount': [24.99, 18.49, 9.99, 18.49, 28.00, 45.00],
})

raw.to_csv(input_csv, index=False)
raw

## Pre-seed persistent memory

Simulates state carried over from previous months — the categorizer will auto-assign these without prompting.

In [ ]:
store = JsonStateStore(state_dir)
store.save_categories(['Electronics', 'Household'])
store.save_mapping({
    'echo dot (3rd gen)':  'Electronics',
    'paper towels 12-pack': 'Household',
    'usb-c cable 6ft':     'Electronics',
})

cats, mapping = load_categorization_state(state_dir)
print('Pre-seeded categories:', cats)
print('Pre-seeded mappings:  ', mapping)

## Run categorizer (non-interactive)

Unknown items are assigned `UNCATEGORIZED`.  In CLI use, `--interactive` prompts for each.

In [ ]:
result, annotated, summary = run_amazon_categorization(
    input_csv_path=input_csv,
    state_dir=state_dir,
    output_dir=output_dir,
)

print(f'Categories known : {result.categories_count}')
print(f'Items mapped     : {result.mapped_items_count}')
print(f'Uncategorized    : {result.uncategorized_count}')

## Annotated output

In [ ]:
annotated

## Category summary

In [ ]:
summary

## Persistent state after run

Unknown items that were actively categorized this run are now remembered.
Items left as `UNCATEGORIZED` are intentionally excluded from the mapping
so they prompt again next month.

In [ ]:
cats, mapping = load_categorization_state(state_dir)
print('Categories:', cats)
print('Mapping:')
for item, cat in sorted(mapping.items()):
    print(f'  {item!r:35} -> {cat}')

## Structured run events

`events.jsonl` is only written when `configure_logging(state_dir)` is called
(done by the CLI).  Here we called `configure_logging()` with no path, so
the frame will be empty — swap in `state_dir` below to see events from real runs.

In [ ]:
events = load_events_dataframe(state_dir)

if events.empty:
    print('No events.jsonl yet — call configure_logging(state_dir) to enable.')
else:
    display(events[['ts', 'event', 'row_count', 'uncategorized_count']])